<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/DailyChallengeW4D5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


In [ ]:

# 1. Chargement des données
pokemon_df = pd.read_csv('C:\\Users\\Debohi\\Downloads\\Pokemon Data Analysis Tutorial\\pokemon.csv')
combats_df = pd.read_csv('C:\\Users\\Debohi\\Downloads\\Pokemon Data Analysis Tutorial\\combats.csv')

# 2. Correction des valeurs manquantes
# Correction du nom pour le Pokémon ayant l'ID/Index 62
pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = 'Primeape'

# Remplacement des NaN dans 'Type 2' par 'Aucun'
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('Aucun')

# 3. Calcul du pourcentage de victoire
# Nombre total de combats par Pokémon (en tant que First_pokemon ou Second_pokemon)
total_combats = combats_df['First_pokemon'].value_counts() + combats_df['Second_pokemon'].value_counts()

# Nombre de victoires par Pokémon
total_victoires = combats_df['Winner'].value_counts()

# Création d'un DataFrame temporaire pour le taux de victoire
win_stats = pd.DataFrame({
    'Total_Combats': total_combats,
    'Victoires': total_victoires
}).fillna(0) # Si un Pokémon n'a jamais gagné ou combattu

win_stats['Win_Percentage'] = (win_stats['Victoires'] / win_stats['Total_Combats']) * 100

# Fusionner le pourcentage de victoire avec le DataFrame principal des Pokémon
pokemon_df = pokemon_df.merge(win_stats['Win_Percentage'], left_on='#', right_index=True, how='left')
# Remplir par 0 si un Pokémon n'a fait aucun combat
pokemon_df['Win_Percentage'] = pokemon_df['Win_Percentage'].fillna(0)

print("Préparation des données terminée avec succès !")

In [ ]:

# Sélection des colonnes statistiques numériques importantes
stats_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Win_Percentage']

# 1. Matrice de corrélation
plt.figure(figsize=(10, 8))
correlation_matrix = pokemon_df[stats_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matrice de corrélation : Statistiques vs Pourcentage de Victoires")
plt.show()

# 2. Pairplot Seaborn
# Pour éviter de surcharger le graphique, on trace un pairgrid ciblé sur Win_Percentage
g = sns.PairGrid(pokemon_df, y_vars=["Win_Percentage"], x_vars=['HP', 'Attack', 'Speed'], height=4)
g.map(sns.regplot, scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.suptitle("Relation entre les stats clés et le % de Victoire", y=1.05)
plt.show()

# 3. Top 10 des meilleurs Pokémon
top_10_pokemon = pokemon_df.sort_values(by='Win_Percentage', ascending=False).head(10)
print("\n--- TOP 10 POKÉMON PAR POURCENTAGE DE VICTOIRE ---")
print(top_10_pokemon[['Name', 'Type 1', 'Speed', 'Attack', 'Win_Percentage']])

In [ ]:

# 1. Définition des Features (X) et de la Cible (y)
# On utilise les statistiques de base pour prédire le pourcentage de victoire
features = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']

X = pokemon_df[features].copy()
# Convertir la colonne booléenne 'Legendary' en numérique (0 ou 1)
X['Legendary'] = X['Legendary'].astype(int)

y = pokemon_df['Win_Percentage']

# 2. Division des données (80% Train / 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialisation des modèles
models = {
    "Régression Linéaire": LinearRegression(),
    "Forêt Aléatoire": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# 4. Entraînement et Évaluation
results_mae = {}

print("\n--- ÉVALUATION DES MODÈLES (MAE) ---")
for name, model in models.items():
    # Entraînement
    model.fit(X_train, y_train)
    # Prédiction
    predictions = model.predict(X_test)
    # Calcul de la MAE
    mae = mean_absolute_error(y_test, predictions)
    results_mae[name] = mae
    print(f"{name} : MAE = {mae:.2f}%")

# 5. Comparaison des performances
best_model = min(results_mae, key=results_mae.get)
print(f"\nLe meilleur modèle est : {best_model} avec la plus petite erreur ({results_mae[best_model]:.2f}%).")